# Tubes PCD: Penggantian Latar Pas Foto (Baseline, Ablasi, Metode Usulan)
Notebook ini untuk Minggu 12 (implementasi) sampai 13 (evaluasi). Isinya:
1. Cek dataset di Drive dan buat `split.json` (stratified, seed tetap)
2. Pipeline modular dengan 4 flag: `use_preprocess`, `use_sobel`, `use_logic_morph`, `use_feathering`
3. Tuning parameter **hanya di tuning set**
4. Evaluasi IoU + waktu per foto di **test set**, simpan CSV dan gambar ke `Project/Hasil/`

**Struktur Drive yang dipakai:** `TUBES PCD/Project/Dataset/{images,masks}` (nama file sama, mis. `img_001.jpg` dan `img_001.png`).
Opsional: `Dataset/metadata.csv` dengan kolom `id,shot,hijab,glasses,env` untuk split stratified.

In [ ]:
import os, json, time, random, itertools
from pathlib import Path
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/TUBES PCD/Project')
except ImportError:  # jalan lokal
    PROJECT = Path(os.environ.get('PCD_PROJECT', './Project'))

DATASET = PROJECT / 'Dataset'
IMG_DIR, MASK_DIR = DATASET / 'images', DATASET / 'masks'
OUT_DIR = PROJECT / 'Hasil'; OUT_DIR.mkdir(parents=True, exist_ok=True)

SIZE = (720, 960)      # (lebar, tinggi), rasio 3:4
SEED = 42
TEST_RATIO = 0.2
TUNE_MAX = 60          # jumlah citra tuning yang dipakai saat grid search (biar cepat)
BG_COLORS = {'merah': '#DB1514', 'biru': '#0090FF'}

## 1. Cek dataset

In [ ]:
EXT = {'.jpg', '.jpeg', '.png'}
imgs  = {p.stem: p for p in sorted(IMG_DIR.iterdir())  if p.suffix.lower() in EXT}
masks = {p.stem: p for p in sorted(MASK_DIR.iterdir()) if p.suffix.lower() in EXT}
labeled = sorted(set(imgs) & set(masks))

print(f'Foto            : {len(imgs)}')
print(f'Mask (GT)       : {len(masks)}')
print(f'Foto + mask     : {len(labeled)}')
print(f'Foto belum label: {len(set(imgs) - set(masks))}')
print(f'Mask tanpa foto : {sorted(set(masks) - set(imgs))[:10]}')

## 2. Split 80:20 (sekali saja, disimpan ke `split.json`)
Split dibuat dari **semua foto** (bukan hanya yang sudah berlabel) supaya stabil walau pelabelan menyusul bertahap.
Kalau `split.json` sudah ada, langsung dipakai. Jangan dihapus kecuali memang mau mengulang dari awal.

In [ ]:
SPLIT_PATH = DATASET / 'split.json'

def make_split():
    meta_path = DATASET / 'metadata.csv'
    if meta_path.exists():
        meta = pd.read_csv(meta_path).set_index('id')
        key = lambda i: tuple(meta.loc[i, ['shot', 'hijab', 'glasses', 'env']]) if i in meta.index else ('na',)
    else:
        print('metadata.csv tidak ada, split acak biasa')
        key = lambda i: ('na',)
    groups = {}
    for i in sorted(imgs):
        groups.setdefault(key(i), []).append(i)
    rng = random.Random(SEED)
    tune, test = [], []
    for g in groups.values():
        rng.shuffle(g)
        n_test = round(len(g) * TEST_RATIO) if len(g) > 1 else 0
        test += g[:n_test]; tune += g[n_test:]
    return {'seed': SEED, 'tune': sorted(tune), 'test': sorted(test)}

if SPLIT_PATH.exists():
    split = json.load(open(SPLIT_PATH))
    print('split.json dimuat')
else:
    split = make_split()
    json.dump(split, open(SPLIT_PATH, 'w'), indent=2)
    print('split.json dibuat')

unassigned = sorted(set(imgs) - set(split['tune']) - set(split['test']))
if unassigned:
    print(f'PERINGATAN: {len(unassigned)} foto baru belum masuk split (hapus split.json kalau mau split ulang)')

TUNE_IDS = [i for i in split['tune'] if i in labeled]
TEST_IDS = [i for i in split['test'] if i in labeled]
print(f'Total split  : tune={len(split["tune"])} test={len(split["test"])}')
print(f'Sudah berlabel: tune={len(TUNE_IDS)} test={len(TEST_IDS)}')

## 3. Loader (crop tengah ke 3:4, resize 720x960, mask di-binarisasi)

In [ ]:
_cache = {}

def fit_3x4(a, interp):
    h, w = a.shape[:2]
    target_w = int(round(h * 3 / 4))
    if w > target_w:                       # terlalu lebar: crop kiri-kanan
        x0 = (w - target_w) // 2; a = a[:, x0:x0 + target_w]
    elif w < target_w:                     # terlalu tinggi: crop atas-bawah
        new_h = int(round(w * 4 / 3)); y0 = (h - new_h) // 2; a = a[y0:y0 + new_h]
    return cv2.resize(a, SIZE, interpolation=interp)

def load_pair(i):
    if i not in _cache:
        img = fit_3x4(cv2.imread(str(imgs[i]), cv2.IMREAD_COLOR), cv2.INTER_AREA)
        gt  = fit_3x4(cv2.imread(str(masks[i]), cv2.IMREAD_GRAYSCALE), cv2.INTER_NEAREST)
        _cache[i] = (img, (gt > 127).astype(np.uint8) * 255)
    return _cache[i]

## 4. Pipeline modular
Alur: preprocessing (CLAHE di kanal L + Gaussian) → color threshold dari sampel warna latar → (opsional) Sobel sebagai *barrier* + flood dari tepi → (opsional) morfologi + isi lubang → (opsional) feathering → alpha blending.

**Skenario**
| Skenario | preprocess | sobel | logic+morph | feathering |
|---|---|---|---|---|
| baseline | - | - | - | - |
| ablasi1 | ✓ | - | ✓ | - |
| ablasi2 | ✓ | ✓ | - | - |
| proposed | ✓ | ✓ | ✓ | ✓ |

In [ ]:
DEFAULT = dict(
    clahe_clip=2.0, clahe_grid=8, blur_k=5,
    space='lab',                 # 'lab' atau 'hsv'
    patch=30,                    # ukuran sampel warna latar (pojok kiri atas, tengah atas, kanan atas)
    tol_l=40, tol_ab=14,         # toleransi LAB (L, a/b)
    tol_h=12, tol_s=60, tol_v=60,  # toleransi HSV (dipakai kalau space='hsv')
    sobel_thr=50, sobel_dilate=3,
    morph_k=7, feather_k=9,
)

SCENARIOS = {
    'baseline': dict(use_preprocess=False, use_sobel=False, use_logic_morph=False, use_feathering=False),
    'ablasi1':  dict(use_preprocess=True,  use_sobel=False, use_logic_morph=True,  use_feathering=False),
    'ablasi2':  dict(use_preprocess=True,  use_sobel=True,  use_logic_morph=False, use_feathering=False),
    'proposed': dict(use_preprocess=True,  use_sobel=True,  use_logic_morph=True,  use_feathering=True),
}

def preprocess(img, p):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=p['clahe_clip'], tileGridSize=(p['clahe_grid'],) * 2)
    lab[..., 0] = clahe.apply(lab[..., 0])          # CLAHE hanya di kanal L
    out = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    k = p['blur_k'] | 1
    return cv2.GaussianBlur(out, (k, k), 0)

def background_mask(img, p):
    """Mask latar (255) dari sampel warna 3 patch di sisi atas citra."""
    n = p['patch']; h, w = img.shape[:2]
    if p['space'] == 'lab':
        conv = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    else:
        conv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    xs = [0, (w - n) // 2, w - n]
    ref = np.median(np.concatenate([conv[0:n, x:x + n].reshape(-1, 3) for x in xs]), axis=0)
    if p['space'] == 'lab':
        tol = np.array([p['tol_l'], p['tol_ab'], p['tol_ab']])
        lo = np.clip(ref - tol, 0, 255).astype(np.uint8)
        hi = np.clip(ref + tol, 0, 255).astype(np.uint8)
        return cv2.inRange(conv, lo, hi)
    dh = np.abs(conv[..., 0].astype(int) - ref[0]); dh = np.minimum(dh, 180 - dh)   # hue melingkar
    ok = (dh <= p['tol_h']) & (np.abs(conv[..., 1] - ref[1]) <= p['tol_s']) & (np.abs(conv[..., 2] - ref[2]) <= p['tol_v'])
    return ok.astype(np.uint8) * 255

def sobel_edges(img, p):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gx = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3)
    mag = cv2.magnitude(gx, gy) / 4.0                 # kira-kira skala 0..360
    e = (mag > p['sobel_thr']).astype(np.uint8) * 255
    k = np.ones((p['sobel_dilate'],) * 2, np.uint8)
    return cv2.dilate(e, k)

def border_connected(mask):
    """Komponen putih yang menyentuh tepi citra (pengganti flood fill dari border)."""
    n, lab = cv2.connectedComponents(mask, connectivity=4)
    edge = np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]])
    ids = np.unique(edge); ids = ids[ids != 0]
    return np.isin(lab, ids).astype(np.uint8) * 255

def clean_foreground(fg, k):
    ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, ker)
    fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, ker)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(fg, connectivity=8)
    if n > 1:                                          # ambil objek terbesar (badan)
        big = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        fg = ((lab == big) * 255).astype(np.uint8)
    inv = cv2.bitwise_not(fg)                          # isi lubang di dalam badan
    holes = cv2.bitwise_and(inv, cv2.bitwise_not(border_connected(inv)))
    return cv2.bitwise_or(fg, holes)

def run_pipeline(img, use_preprocess, use_sobel, use_logic_morph, use_feathering, **override):
    p = {**DEFAULT, **override}
    work = preprocess(img, p) if use_preprocess else img
    bg = background_mask(work, p)
    if use_sobel:
        barrier = sobel_edges(work, p)
        bg = border_connected(cv2.bitwise_and(bg, cv2.bitwise_not(barrier)))   # latar tidak boleh menembus tepi Sobel
    fg = cv2.bitwise_not(bg)
    if use_logic_morph:
        fg = clean_foreground(fg, p['morph_k'])
    if use_feathering:
        k = p['feather_k'] | 1
        alpha = cv2.GaussianBlur(fg.astype(np.float32) / 255.0, (k, k), 0)
    else:
        alpha = fg.astype(np.float32) / 255.0
    return fg, alpha

def replace_bg(img, alpha, hex_color):
    rgb = [int(hex_color[i:i + 2], 16) for i in (1, 3, 5)]
    bgr = np.array(rgb[::-1], np.float32)
    a = alpha[..., None]
    return (a * img + (1 - a) * bgr).astype(np.uint8)

def iou(pred, gt):
    p, g = pred > 127, gt > 127
    union = np.logical_or(p, g).sum()
    return 1.0 if union == 0 else float(np.logical_and(p, g).sum() / union)

## 5. Tuning (hanya tuning set, prosedur sama untuk semua skenario)

In [ ]:
def tune(name, ids, n_max=TUNE_MAX):
    flags = SCENARIOS[name]
    grid = {'tol_l': [25, 40, 55], 'tol_ab': [8, 14, 20]}
    if flags['use_sobel']:
        grid['sobel_thr'] = [30, 50, 80]
    ids = sorted(ids)
    if len(ids) > n_max:
        ids = random.Random(SEED).sample(ids, n_max)
    best_score, best_p = -1, None
    for combo in itertools.product(*grid.values()):
        p = dict(zip(grid, combo))
        s = np.mean([iou(run_pipeline(load_pair(i)[0], **flags, **p)[0], load_pair(i)[1]) for i in ids])
        if s > best_score:
            best_score, best_p = s, p
    return best_p, best_score

assert len(TUNE_IDS) > 0, 'Belum ada citra tuning yang punya mask. Isi dulu images/ dan masks/.'
BEST = {}
for name in SCENARIOS:
    BEST[name], sc = tune(name, TUNE_IDS)
    print(f'{name:9s} IoU tuning={sc:.4f}  param={BEST[name]}')
json.dump(BEST, open(OUT_DIR / 'best_params.json', 'w'), indent=2)

## 6. Evaluasi di test set (IoU + waktu per foto)

In [ ]:
def evaluate(ids):
    rows = []
    for i in ids:
        img, gt = load_pair(i)
        row = {'id': i}
        for name, flags in SCENARIOS.items():
            t0 = time.perf_counter()
            fg, alpha = run_pipeline(img, **flags, **BEST[name])
            _ = replace_bg(img, alpha, BG_COLORS['biru'])
            row[f'iou_{name}'] = iou(fg, gt)
            row[f'ms_{name}'] = (time.perf_counter() - t0) * 1000
        rows.append(row)
    return pd.DataFrame(rows)

assert len(TEST_IDS) > 0, 'Belum ada citra test yang punya mask.'
res = evaluate(TEST_IDS)
res.to_csv(OUT_DIR / 'hasil_per_citra.csv', index=False)

base = res['iou_baseline'].mean()
summary = pd.DataFrame({
    'IoU_mean': [res[f'iou_{n}'].mean() for n in SCENARIOS],
    'IoU_std':  [res[f'iou_{n}'].std() for n in SCENARIOS],
    'waktu_ms': [res[f'ms_{n}'].mean() for n in SCENARIOS],
}, index=list(SCENARIOS))
summary['vs_baseline_%'] = (summary['IoU_mean'] / base - 1) * 100
summary.round(4).to_csv(OUT_DIR / 'ringkasan.csv')
display(summary.round(4))

print(f'Target IoU >= 0.80 (proposed)         : {"OK" if summary.loc["proposed","IoU_mean"] >= 0.80 else "belum"}')
print(f'Target +15% vs baseline               : {"OK" if summary.loc["proposed","vs_baseline_%"] >= 15 else "belum"}')
print(f'Target < 2 detik/foto (proposed)      : {"OK" if summary.loc["proposed","waktu_ms"] < 2000 else "belum"}')

## 7. Visualisasi dan analisis kasus gagal

In [ ]:
def show_compare(ids, color='biru', save=None):
    names = list(SCENARIOS)
    fig, ax = plt.subplots(len(ids), 2 + len(names), figsize=(3 * (2 + len(names)), 4 * len(ids)), squeeze=False)
    for r, i in enumerate(ids):
        img, gt = load_pair(i)
        rgb = lambda a: cv2.cvtColor(a, cv2.COLOR_BGR2RGB)
        ax[r, 0].imshow(rgb(img)); ax[r, 0].set_title(i)
        ax[r, 1].imshow(rgb(replace_bg(img, gt / 255.0, BG_COLORS[color]))); ax[r, 1].set_title('GT')
        for c, n in enumerate(names, start=2):
            fg, alpha = run_pipeline(img, **SCENARIOS[n], **BEST[n])
            ax[r, c].imshow(rgb(replace_bg(img, alpha, BG_COLORS[color])))
            ax[r, c].set_title(f'{n} IoU={iou(fg, gt):.3f}')
    for a in ax.ravel(): a.axis('off')
    plt.tight_layout()
    if save: fig.savefig(save, dpi=120)
    plt.show()

sample = random.Random(SEED).sample(TEST_IDS, min(4, len(TEST_IDS)))
show_compare(sample, save=OUT_DIR / 'perbandingan_sampel.png')

worst = res.nsmallest(4, 'iou_proposed')[['id', 'iou_baseline', 'iou_ablasi1', 'iou_ablasi2', 'iou_proposed']]
display(worst.round(4))
show_compare(list(worst['id']), save=OUT_DIR / 'kasus_gagal.png')

## 8. Langkah berikutnya
- Zoom ke area **rambut dan bahu** untuk bahas Rumusan Masalah 3 (kualitas batas, bukan hanya IoU). Bisa tambah Boundary IoU kalau waktu cukup.
- Kelompokkan hasil per kategori (hijab, kacamata, indoor/outdoor) pakai `metadata.csv` untuk analisis kasus gagal.
- `run_pipeline()` + `replace_bg()` langsung dipakai di Streamlit (Minggu 14). Pindahkan ke `src/` setelah stabil.
- Setelah dataset sudah penuh, jalankan ulang dari atas. Tuning dan evaluasi otomatis memakai `split.json` yang sama.